In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# ===========================================================================
# Notebook  : bronze_user
# Location  : /HGI-Lakehouse-Pipeline/02_Bronze_Layer/bronze_user
# Purpose   : SALESFORCE User → bronze.{customer_id}.crm_users
#
# Serverless : Set max_files_per_trigger=10 for Serverless 2X-Small testing
# Job Compute: Set max_files_per_trigger=200 for production
# ===========================================================================

In [0]:

# CELL 1 ── Load shared config
# In Databricks: uncomment the two %run lines below.
# Each %run MUST be in its own cell (magic commands require their own cell).
%run ../config/pipeline_config.py


In [0]:
%run ./bronze_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2 ── Widgets
dbutils.widgets.text("customer_id",           "")
dbutils.widgets.text("load_type",             "")   # historical | incremental
dbutils.widgets.text("max_files_per_trigger", "200")
# Serverless tip: override max_files_per_trigger to '10' on Serverless 2X-Small

customer_id           = dbutils.widgets.get("customer_id").strip().lower()
load_type             = dbutils.widgets.get("load_type").strip().lower()
max_files_per_trigger = int(dbutils.widgets.get("max_files_per_trigger"))

In [0]:
# CELL 3 ── Object-specific constants
source_system  = "salesforce"
object_name    = "user"
sf_object_name = "User"
# Composite ID will be: salesforce_User_Id_{SalesforceId}
tenant_id = tenant_id_from_customer(customer_id)   # helper from pipeline_config

In [0]:
import sys, os
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
if os.path.abspath(project_root) not in sys.path:
    sys.path.append(os.path.abspath(project_root))

from utils.pipeline_metrics import PipelineMetrics
pm = PipelineMetrics(
    spark          = spark,
    parent_run_id  = parent_run_id,
    job_name       = "bronze_user",
    task_key       = "run_job_C2_sf_bronze",
    source_system  = source_system,
    load_type      = load_type,
    customer_id    = customer_id,
    object_name    = object_name,
)
pm.init()

# Ensure audit tables exist for error logging
initialize_audit_tables()

In [0]:
# CELL 4 -- Path resolution  (all helpers from pipeline_config)
landing_zone_path = landing_path(source_system, customer_id, object_name, load_type)
checkpoint_loc    = checkpoint_path("bronze", source_system, customer_id, object_name) + load_type + "/"  # Bug 1 fix
schema_loc        = checkpoint_loc + "_schema/"
table_full_name   = bronze_table(customer_id, object_name)

print(f"=== Bronze: SALESFORCE User ===")
print(f"  Customer   : {customer_id}  (tenant={tenant_id})")
print(f"  Load type  : {load_type}")
print(f"  Landing    : {landing_zone_path}")
print(f"  Checkpoint : {checkpoint_loc}")
print(f"  Target     : {table_full_name}")
print(f"  Batch size : {max_files_per_trigger} files/trigger")

In [0]:
# CELL 5 ── Create bronze schema + table
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_catalog}.{customer_id}")
create_bronze_table(table_full_name, object_name)  # uses BRONZE_DDL_COLUMNS['user']

In [0]:
# CELL 6 ── Run pipeline
try:
    dbutils.fs.ls(landing_zone_path)
except Exception:
    print(f"⚠️ No data at landing zone path: {landing_zone_path}")
    print("Skipping ingestion — no files to process.")
    pm.save(status="SUCCESS", rows_processed=0)
    dbutils.notebook.exit("NO_DATA")

try:
    load_summary = run_with_retry(start_ingestion)
    log_audit(
        job_name       = f"bronze_{object_name}",
        customer_id    = customer_id,
        status         = "success",
        layer          = "bronze",
        alert_type     = "SUCCESS",
        rows_processed = bronze_logger.total_merged,
    )
except Exception as e:
    print(f"❌ Pipeline failed: {e}")
    log_audit(
        job_name       = f"bronze_{object_name}",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "bronze",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
# CELL 7 ── Post-load maintenance
try:
    print("Running OPTIMIZE + VACUUM ...")
    spark.sql(f"OPTIMIZE {table_full_name}")
    spark.sql(f"VACUUM {table_full_name} RETAIN 168 HOURS")
    print(f"Bronze load complete: {table_full_name}")
except Exception as e:
    print(f"❌ Post-load maintenance failed: {e}")
    log_audit(
        job_name       = f"bronze_{object_name}",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "bronze",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = f"OPTIMIZE/VACUUM failed: {str(e)[:450]}",
    )
    raise

In [0]:
try:
    total_rows = spark.table(table_full_name).count()
    pm.save(status="SUCCESS", rows_processed=total_rows)
    print(f"\n\u2705 Pipeline metrics saved: {total_rows:,} total rows in {table_full_name}")
    print(f"   CDC records merged this run: {bronze_logger.total_merged:,}")
    print(f"   Records from landing: {bronze_logger.total_landing:,}")
    print(f"   Duplicates removed: {bronze_logger.total_landing - bronze_logger.total_deduped:,}")
    print(f"   Unchanged skipped: {bronze_logger.total_deduped - bronze_logger.total_merged:,}")
except Exception as e:
    print(f"\u274c Metrics save failed: {e}")
    log_audit(
        job_name       = f"bronze_{object_name}",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "bronze",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise